## make sure to get a sas token for the blob container

https://portal.azure.com/#@jordanryderlive.onmicrosoft.com/resource/subscriptions/8d99516f-8acf-4147-a8d6-7e0aa5c7af85/resourceGroups/resource_personal/providers/Microsoft.Storage/storageAccounts/storagejordanryder/storagebrowser


In [ ]:
import json
import os
from os import path
from azure.identity import DefaultAzureCredential, AzureCliCredential
from azure.storage.blob import BlobServiceClient, ContainerClient, BlobPrefix
from Api_Helpers import DbConnector

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

container_name = os.environ["AZURE_STORAGE_CONTAINER"]
account_url = os.environ["AZURE_STORAGE_ACCOUNT_URL"]
sas_token = os.environ["AZURE_STORAGE_SAS_TOKEN"]

# make sure to get a sas token for the blob container

blob_service_client = BlobServiceClient(f"{account_url}/?{sas_token}")
container_client = blob_service_client.get_container_client(container_name)

db = DbConnector()


In [ ]:

#blobs = container_client.list_blobs()
#for blob in blobs:
#    print(blob.name)
#    if not db.exists(blob.name):
#        db.add_path(blob.name)


chunk_size=4*1024*1024
limit = 800*1024*1024

def iterate(filepath:str):
    for file in os.listdir(filepath):
        
        fullpath = path.join(filepath,file)
        if path.isfile(fullpath):
            upload_full_path = fullpath[12:]
            file_modified = os.path.getmtime(fullpath) 
            # print(fullpath)
            if db.exists_path(upload_full_path, file_modified):
                # print(upload_full_path)
                print('-', end='')
                continue
            
            if path.getsize(fullpath) > limit: #200 mb
                print('/',end='')
                continue
            if 'Documents/Games/' in fullpath:
                print('-')
                continue
            print(upload_full_path)
            if path.getsize(fullpath) > chunk_size:
                block_ids = []
                blob_client = blob_service_client.get_blob_client(container=container_name, blob=upload_full_path)

                with open(fullpath, "rb") as f:
                    while True:
                        chunk = f.read(chunk_size)
                        if not chunk:
                            break
                        
                        block_id = str(len(block_ids)).zfill(6)  # Generate block IDs
                        block_ids.append(block_id)
                        
                        blob_client.stage_block(block_id=block_id, data=chunk)
                blob_client.commit_block_list(block_ids)
            else:
                with open(file=fullpath, mode="rb") as data:
                    container_client.upload_blob(name=upload_full_path, data=data, overwrite=True)
            db.add_path(upload_full_path, file_modified)
        elif file[0] != '.' and 'NetBeans' not in file and 'actions-runner' not in file and 'Documents/Games/' not in fullpath:
            iterate(fullpath)


iterate('/home/jordan/Pictures')
iterate('/home/jordan/Documents')


